# Convolutional Neural Networks

## Data loading and analysis

In [ ]:
import gc
import timm
import joblib
import cv2
import torch
import hashlib
import numpy as np
import pandas as pd

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from timm.models import resnet18

from torchvision import transforms
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

In [ ]:
data = pd.read_csv("/content/drive/MyDrive/21 school/ML9_dataset/ML9_dataset/Train.csv")
data = data.set_index('img_IDS')

In [ ]:
hashes = dict()
im_ids = np.asarray(data.index)
labels = np.asarray(data["Label"])
unique_labels = list(set(labels))
duplicates = []
base_path = "/content/drive/MyDrive/21 school/ML9_dataset/ML9_dataset/Images/"

In [ ]:
def get_file_hash(image_path):
    hash_md5 = hashlib.md5()
    with open(image_path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

for id in im_ids:
    impath = base_path + id + ".jpg"
    hash = get_file_hash(impath)
    try:
        hashes[hash].append(id)
        duplicates.append(id)
    except KeyError:
        hashes[hash] = [id]

print("Дубликатов: ", len(duplicates))
print("Значений знаков: ", len(unique_labels))

Дубликатов:  0
Значений знаков:  9


In [ ]:
unique_labels = np.unique(labels)
label_to_id = {label: i for i, label in enumerate(unique_labels)}
data["Label"] = data["Label"].apply(lambda elem: label_to_id[elem])

In [ ]:
train, val = train_test_split(im_ids, test_size=1/3, random_state=21)
train = data.loc[train]
val = data.loc[val]

In [ ]:
del data, hashes, im_ids, labels
gc.collect()

0

## 2.Dataset class

In [ ]:
class SignLanguageDataset(Dataset):
    def __init__(self, data, base_path, transform=None):
        self.data = data.reset_index()
        self.data = self.data.to_numpy()
        self.base_path = base_path
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, ind):
        img_id, label = self.data[ind]
        img_path = f"{self.base_path}{img_id}.jpg"

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            try:
                image = self.transform(image=image)
                image = image['image']
            except TypeError:
                image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

In [ ]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = SignLanguageDataset(train, base_path, transform=transform)
val_dataset = SignLanguageDataset(val, base_path, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
bb_train = train.sample(200, random_state=21)
bb_val = val.sample(200, random_state=21)

bb_train_dataset = SignLanguageDataset(bb_train, base_path, transform=transform)
bb_val_dataset = SignLanguageDataset(bb_val, base_path, transform=transform)

bb_train_loader = DataLoader(bb_train_dataset, batch_size=10, shuffle=True)
bb_val_loader = DataLoader(bb_val_dataset, batch_size=10, shuffle=False)

## 3.LeNet-5

In [ ]:
class LeNet5(nn.Module):
    def __init__(self, num_classes, imsize):
        super(LeNet5, self).__init__()
        self.res_size = (((imsize - 4) // 2) - 4) // 2

        self.conv1 = nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5, stride=1, padding=0)
        self.pool1 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1, padding=0)
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)

        self.dropout = nn.Dropout(p=0.5)
        self.fc1 = nn.Linear(in_features=16 * self.res_size[0] * self.res_size[1], out_features=120)
        self.fc2 = nn.Linear(in_features=120, out_features=84)
        self.fc3 = nn.Linear(in_features=84, out_features=num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool2(x)

        x = x.view(-1, 16 * self.res_size[0] * self.res_size[1])
        x = self.fc1(x)
        x = F.leaky_relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = F.leaky_relu(x)
        x = self.dropout(x)
        x = self.fc3(x)
        x = F.softmax(x, dim=1)
        return x

## 4.Train-val loop and model estimations

In [ ]:
def train_model(model, train_loader, optimizer, epoch, log_interval):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()

        if batch_idx % log_interval == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))


def test_model(model, test_loader):
    model.eval()
    test_loss = 0
    score_av = 0
    counter = 0
    with torch.no_grad():
        for data, target in test_loader:
            counter += 1
            output = model(data)
            test_loss += F.cross_entropy(output, target, reduction='sum').item()
            score = roc_auc_score(np.asarray(target), np.asarray(output), multi_class='ovo', labels=[0,1,2,3,4,5,6,7,8])
            score_av += score

    test_loss /= len(test_loader.dataset)
    score_av /= counter

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}\n'.format(
        test_loss, score_av))

In [ ]:
def main(train_loader, val_loader, imdim, epochs):
    torch.manual_seed(21)

    model = LeNet5(len(unique_labels), imdim)
    optimizer = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.999))

    scheduler = StepLR(optimizer, step_size=1, gamma=0.7)
    for epoch in range(1, epochs + 1):
        train_model(model, train_loader, optimizer, epoch, 100)
        test_model(model, val_loader)
        scheduler.step()

In [ ]:
main(train_loader, val_loader, np.array([224, 224]), 10)

Train Epoch: 1 [0/4166 (0%)]	Loss: 2.199600
Train Epoch: 1 [3200/4166 (76%)]	Loss: 2.197426

Test set: Average loss: 2.1936, Accuracy: 0.596087211797364

Train Epoch: 2 [0/4166 (0%)]	Loss: 2.192024
Train Epoch: 2 [3200/4166 (76%)]	Loss: 2.202357

Test set: Average loss: 2.1418, Accuracy: 0.6216018993116795

Train Epoch: 3 [0/4166 (0%)]	Loss: 2.099730
Train Epoch: 3 [3200/4166 (76%)]	Loss: 2.101108

Test set: Average loss: 2.0847, Accuracy: 0.6734650118874245

Train Epoch: 4 [0/4166 (0%)]	Loss: 2.083778
Train Epoch: 4 [3200/4166 (76%)]	Loss: 2.112125

Test set: Average loss: 2.0471, Accuracy: 0.7112201224777699

Train Epoch: 5 [0/4166 (0%)]	Loss: 2.102617
Train Epoch: 5 [3200/4166 (76%)]	Loss: 2.012424

Test set: Average loss: 2.0242, Accuracy: 0.7342501339293666

Train Epoch: 6 [0/4166 (0%)]	Loss: 2.041117
Train Epoch: 6 [3200/4166 (76%)]	Loss: 1.978658

Test set: Average loss: 2.0077, Accuracy: 0.7489423949235042

Train Epoch: 7 [0/4166 (0%)]	Loss: 2.056716
Train Epoch: 7 [3200/4166 (

## 5.Other models

In [ ]:
class Head(nn.Module):
    def __init__(self, input_features, num_classes):
        super(Head, self).__init__()

        self.dropout = nn.Dropout(p=0.5)
        self.fc1 = nn.Linear(input_features, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = F.leaky_relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = F.leaky_relu(x)
        x = self.dropout(x)
        x = self.fc3(x)
        x = F.softmax(x, dim=1)

        return x

class FullModel(nn.Module):
    def __init__(self, backbone, head):
        super(FullModel, self).__init__()
        self.backbone = backbone
        self.head = head

    def forward(self, data):
        features = self.backbone(data)

        features = features.view(features.size(0), -1)

        output = self.head(features)
        return output

In [ ]:
def main_test(train_loader, val_loader, epochs, model):
    torch.manual_seed(21)

    optimizer = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.999))

    scheduler = StepLR(optimizer, step_size=1, gamma=0.7)
    for epoch in range(1, epochs + 1):
        train_model(model, train_loader, optimizer, epoch, 50)
        test_model(model, val_loader)
        scheduler.step()

In [ ]:
%%capture
resnet_model = resnet18(num_classes=9, pretrained=True)

In [ ]:
input_features = resnet_model.fc.in_features
resnet_model.fc = nn.Identity()

head = Head(input_features, 9)
full_model = FullModel(resnet_model, head)

In [ ]:
main_test(train_loader, val_loader, 4, full_model)

Train Epoch: 1 [0/4166 (0%)]	Loss: 2.198853
Train Epoch: 1 [1600/4166 (38%)]	Loss: 2.192938
Train Epoch: 1 [3200/4166 (76%)]	Loss: 1.986888

Test set: Average loss: 1.9136, Accuracy: 0.8839659435077342

Train Epoch: 2 [0/4166 (0%)]	Loss: 1.918216
Train Epoch: 2 [1600/4166 (38%)]	Loss: 1.697754
Train Epoch: 2 [3200/4166 (76%)]	Loss: 1.819598

Test set: Average loss: 1.5528, Accuracy: 0.974717369270941

Train Epoch: 3 [0/4166 (0%)]	Loss: 1.651368
Train Epoch: 3 [1600/4166 (38%)]	Loss: 1.578840
Train Epoch: 3 [3200/4166 (76%)]	Loss: 1.518988

Test set: Average loss: 1.5094, Accuracy: 0.986333951404636

Train Epoch: 4 [0/4166 (0%)]	Loss: 1.435775
Train Epoch: 4 [1600/4166 (38%)]	Loss: 1.464085
Train Epoch: 4 [3200/4166 (76%)]	Loss: 1.489670

Test set: Average loss: 1.4867, Accuracy: 0.9872894960804189



In [ ]:
joblib.dump(full_model, '/content/drive/MyDrive/21 school/torch-afro-vision.joblib')

['/content/drive/MyDrive/21 school/torch-afro-vision.joblib']

In [ ]:
gc.collect()

227

## 6. Augmentations

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [ ]:
aug_transform = A.Compose([
    A.Resize(height=224, width=224),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

test_aug_transform = A.Compose([
    A.Resize(height=224, width=224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

In [ ]:
aug_transform = A.Compose([
    A.Resize(height=224, width=224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),

    A.RandomRotate90(),
    A.HorizontalFlip(p=0.5),

    A.OneOf([
        A.MotionBlur(p=0.5),
        A.MedianBlur(blur_limit=3, p=0.5),
        A.GaussianBlur(blur_limit=3, p=0.5),
    ]),

    ToTensorV2()
])

In [ ]:
train_dataset = SignLanguageDataset(train, base_path, transform=aug_transform)
val_dataset = SignLanguageDataset(val, base_path, transform=test_aug_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
full_model = FullModel(resnet_model, head)
main_test(train_loader, val_loader, 4, full_model)

Train Epoch: 1 [0/4166 (0%)]	Loss: 2.199013
Train Epoch: 1 [1600/4166 (38%)]	Loss: 2.210797
Train Epoch: 1 [3200/4166 (76%)]	Loss: 2.203378

Test set: Average loss: 2.2205, Accuracy: 0.5608335571122856

Train Epoch: 2 [0/4166 (0%)]	Loss: 2.121886
Train Epoch: 2 [1600/4166 (38%)]	Loss: 2.113663
Train Epoch: 2 [3200/4166 (76%)]	Loss: 2.218148

Test set: Average loss: 2.0267, Accuracy: 0.7747198539032619

Train Epoch: 3 [0/4166 (0%)]	Loss: 2.149189
Train Epoch: 3 [1600/4166 (38%)]	Loss: 2.011026
Train Epoch: 3 [3200/4166 (76%)]	Loss: 1.855878

Test set: Average loss: 1.9526, Accuracy: 0.8296417105118327

Train Epoch: 4 [0/4166 (0%)]	Loss: 1.997509
Train Epoch: 4 [1600/4166 (38%)]	Loss: 1.943986
Train Epoch: 4 [3200/4166 (76%)]	Loss: 1.859097

Test set: Average loss: 1.8346, Accuracy: 0.8655692777485109



Дополнительные аугментации общий скор не улучшают

## 7. MixUp and CutMix

In [ ]:
def mixup_data(x, y, alpha=1.0):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]

    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [ ]:
def train_epoch_mixup(model, train_loader, optimizer, epoch, log_interval, alpha=0.6):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, targets_a, targets_b, lam = mixup_data(data, target, alpha)
        optimizer.zero_grad()
        output = model(data)
        loss = mixup_criterion(F.cross_entropy, output, targets_a, targets_b, lam)
        loss.backward()
        optimizer.step()

        if batch_idx % log_interval == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))

In [ ]:
def main_test(train_loader, val_loader, epochs, model):
    torch.manual_seed(21)

    optimizer = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.999))

    scheduler = StepLR(optimizer, step_size=1, gamma=0.7)
    for epoch in range(1, epochs + 1):
        train_epoch_mixup(model, train_loader, optimizer, epoch, 50)
        test_model(model, val_loader)
        scheduler.step()

In [ ]:
full_model = FullModel(resnet_model, head)
main_test(train_loader, val_loader, 4, full_model)

Train Epoch: 1 [0/4166 (0%)]	Loss: 2.197055
Train Epoch: 1 [1600/4166 (38%)]	Loss: 2.145887
Train Epoch: 1 [3200/4166 (76%)]	Loss: 2.098234

Test set: Average loss: 1.9523, Accuracy: 0.843381814431955

Train Epoch: 2 [0/4166 (0%)]	Loss: 2.127443
Train Epoch: 2 [1600/4166 (38%)]	Loss: 1.846774
Train Epoch: 2 [3200/4166 (76%)]	Loss: 1.951357

Test set: Average loss: 1.7299, Accuracy: 0.9273795345509135

Train Epoch: 3 [0/4166 (0%)]	Loss: 1.878627
Train Epoch: 3 [1600/4166 (38%)]	Loss: 1.748379
Train Epoch: 3 [3200/4166 (76%)]	Loss: 1.534019

Test set: Average loss: 1.5652, Accuracy: 0.973284352882567

Train Epoch: 4 [0/4166 (0%)]	Loss: 1.618178
Train Epoch: 4 [1600/4166 (38%)]	Loss: 1.528332
Train Epoch: 4 [3200/4166 (76%)]	Loss: 1.784407

Test set: Average loss: 1.5252, Accuracy: 0.9800792088019269



In [ ]:
joblib.dump(full_model, '/content/drive/MyDrive/21 school/torch-afro-vision-MixUp.joblib')

['/content/drive/MyDrive/21 school/torch-afro-vision-MixUp.joblib']

In [ ]:
def rand_bbox(size, lam):
    W = size[2]
    H = size[3]

    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    cx = np.random.randint(W)
    cy = np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2

def cutmix_data(x, y, alpha=0.6):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)

    mixed_x = x.clone()
    mixed_x[:, :, bbx1:bbx2, bby1:bby2] = x[index, :, bbx1:bbx2, bby1:bby2]

    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (x.size(-1) * x.size(-2)))
    y_a, y_b = y, y[index]

    return mixed_x, y_a, y_b, lam

In [ ]:
def train_epoch_cutmix(model, train_loader, optimizer, epoch, log_interval, alpha=0.6):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, targets_a, targets_b, lam = cutmix_data(data, target, alpha)
        optimizer.zero_grad()
        output = model(data)
        loss = mixup_criterion(F.cross_entropy, output, targets_a, targets_b, lam)
        loss.backward()
        optimizer.step()

        if batch_idx % log_interval == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))

In [ ]:
def main_test(train_loader, val_loader, epochs, model):
    torch.manual_seed(21)

    optimizer = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.999))

    scheduler = StepLR(optimizer, step_size=1, gamma=0.7)
    for epoch in range(1, epochs + 1):
        train_epoch_cutmix(model, train_loader, optimizer, epoch, 50)
        test_model(model, val_loader)
        scheduler.step()

In [ ]:
full_model = FullModel(resnet_model, head)
main_test(train_loader, val_loader, 4, full_model)

Train Epoch: 1 [0/4166 (0%)]	Loss: 2.197920
Train Epoch: 1 [1600/4166 (38%)]	Loss: 2.181384
Train Epoch: 1 [3200/4166 (76%)]	Loss: 2.156649

Test set: Average loss: 2.0659, Accuracy: 0.7383438623766008

Train Epoch: 2 [0/4166 (0%)]	Loss: 2.019147
Train Epoch: 2 [1600/4166 (38%)]	Loss: 2.056026
Train Epoch: 2 [3200/4166 (76%)]	Loss: 2.096629

Test set: Average loss: 1.8047, Accuracy: 0.923770352452148

Train Epoch: 3 [0/4166 (0%)]	Loss: 2.030836
Train Epoch: 3 [1600/4166 (38%)]	Loss: 1.992415
Train Epoch: 3 [3200/4166 (76%)]	Loss: 1.811285

Test set: Average loss: 1.5979, Accuracy: 0.9693479377581559

Train Epoch: 4 [0/4166 (0%)]	Loss: 1.941514
Train Epoch: 4 [1600/4166 (38%)]	Loss: 1.791468
Train Epoch: 4 [3200/4166 (76%)]	Loss: 1.925369

Test set: Average loss: 1.5191, Accuracy: 0.9807278782725212



Both MixUp and CutMix do not improve validation score, though they get almost the same results as the baseline model. But I also should notice that the loss difference is more gradual and stable with the MixUp than for the baseline model. This can show us that the MixUp is more stable at training and less prone to overfitting, though overall score is lower.

## 8. TTA

In [ ]:
def test_model(model, test_loader, transforms_list):
    model.eval()
    test_loss = 0
    score_av = 0
    counter = 0
    with torch.no_grad():
        for data, target in test_loader:
            counter += len(transforms_list)
            for t in transforms_list:
                data = t(data)
                output = model(data)
                test_loss += F.cross_entropy(output, target, reduction='sum').item()
                score = roc_auc_score(np.asarray(target), np.asarray(output), multi_class='ovo', labels=[0,1,2,3,4,5,6,7,8])
                score_av += score

    test_loss /= len(transforms_list) * len(test_loader.dataset)
    score_av /= counter

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}\n'.format(
        test_loss, score_av))

In [ ]:
full_model = joblib.load('/content/drive/MyDrive/21 school/torch-afro-vision.joblib')

transforms_list = [
    lambda x: x,
    lambda x: transforms.functional.hflip(x)
]

In [ ]:
test_model(full_model, val_loader, transforms_list)


Test set: Average loss: 1.5130, Accuracy: 0.9813694290672219



In [ ]:
full_model = joblib.load('/content/drive/MyDrive/21 school/torch-afro-vision-MixUp.joblib')

In [ ]:
test_model(full_model, val_loader, transforms_list)


Test set: Average loss: 1.5585, Accuracy: 0.9727666355543342



The score from the Test Time Augmentation drop a little bit(1%<), which is normal variance, but also it can hint us that the model is close to or already is optimal, meaning it is invariant to horizontal flip.